# **BUSINESS DATA ANALYSTS (From Nguyen Anh Tuan)**

## **1. Tổng quan kinh doanh** 

**Nhu cầu:**
- **Đối tượng sử dụng:** CEO, COO, Sales Director, Business Analysts
- **Mục đích:** Nắm bắt nhanh tình hình kinh doanh tổng thể trong 1 góc nhìn
- **Tần suất xem:** Hàng ngày, real-time
- **Câu hỏi kinh doanh:**
    - Doanh thu hiện tại đang ở mức nào?
    - Có bao nhiêu đơn hàng đã được xử lý?
    - Khách hàng mua trung bình bao nhiêu tiền mỗi đơn?
    - Số lượng sản phẩm bán ra là bao nhiêu?

**Source Data**

<p align="center">
  <img src="images/data_structure.png" alt="Dashboard sales" width="500">
</p>

**Output Dashboard:**
- 4 KPI Cards chính:
    - 💰 **Total Revenue:** 66,272,500 VND
    - 📦 **Total Orders:** 30
    - 🛒 **Total Items Sold:** 135
    - 💵 **Average Order Value:** 2,209,083 VND

## **2. Metrics Brakdown**

### **2.1. Metric 1: Total Revenue**

**Công thức:**

```sql
SUM(quantity × unit_price × (1 - discount_percent/100))

**Business Logic:**
- Revenue = Số lượng × Đơn giá × (1 - Chiết khấu)
- Tính tổng tất cả order details
- Đã trừ đi chiết khấu để có net revenue

**Insights:**
- Chỉ số quan trọng nhất đo lường quy mô kinh doanh
- So sánh với mục tiêu để đánh giá performance
- Track theo time để thấy growth trend

### **2.2. Metric 2: Total Order**

**Công thức:**

```sql
COUNT(DISTINCT order_id)

**Business Logic:**
- Đếm số lượng đơn hàng unique
- Không trùng lặp (DISTINCT)

**Insights:**

- Đo lường khối lượng giao dịch
- Kết hợp với revenue để tính AOV
- Indicator về customer acquisition

### **2.3. Metric 3: Total Items Sold**

**Công thức:**

```sql
SUM(quantily)

**Business Logic:**
- Cộng tổng số lượng tất cả sản phẩm
- Khác với số order (1 order có nhiều items)

**Insights:**
- Đo lường lưu lượng hàng hóa
- Dùng cho inventory planning
- Indicator về cross-selling effectiveness

### **2.4. Metric 4: Average Order Value**

**Công thức:**

```sql
Total revenue / Total orders

**Business Logic:**
- AOV = Tổng doanh thu chia cho số đơn hàng
- Derived metric từ 2 metrics trên

**Insights:**
- Chỉ số quan trọng cho pricing strategy
- Target cho upselling/cross-selling campaigns
- Benchmark để so sánh với industry average

## 3. Data lineage & Implementation

<div align="center">

<table>
  <tr>
    <td align="center">
      <b>Layer 1</b><br>
      Data Source<br>
      <i>Nguồn dữ liệu</i>
    </td>
  </tr>
</table>

⬇️

<table>
  <tr>
    <td align="center">
      <b>Layer 2</b><br>
      Base Analytics Table<br>
      <i>Bảng phân tích cơ sở</i>
    </td>
  </tr>
</table>

⬇️

<table>
  <tr>
    <td align="center">
      <b>Layer 3</b><br>
      Aggregation Tables<br>
      <i>Bảng tổng hợp</i>
    </td>
  </tr>
</table>

⬇️

<table>
  <tr>
    <td align="center">
      <b>Layer 4</b><br>
      KPI Metrics<br>
      <i>Chỉ số hiển thị</i>
    </td>
  </tr>
</table>

⬇️

<table>
  <tr>
    <td align="center">
      <b>Layer 5</b><br>
      Dashboard Visualization<br>
      <i>Trực quan hóa</i>
    </td>
  </tr>
</table>

</div>

## **4. Implementation Details**

### **Step 1: Import package**

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### **Step 2: Import data source & Exprort xlsx to CSV**

In [7]:
from pathlib import Path

def export_xlsx_sheets_to_csv(xlsx_path, output_dir):
    xlsx_path = Path(xlsx_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    all_sheets = pd.read_excel(xlsx_path, sheet_name=None)
    created_files = []

    for sheet_name, df in all_sheets.items():
        safe_sheet_name = ''.join(char if char.isalnum() or char in (' ', '-', '_') else '_' for char in sheet_name).strip()
        csv_path = output_dir / f'{safe_sheet_name}.csv'
        df.to_csv(csv_path, index=False)
        created_files.append(csv_path)
        print(f'Saved: {csv_path.name}')

    return all_sheets, created_files

all_sheets, created_files = export_xlsx_sheets_to_csv('data/Retail.xlsx', 'data')
print('Sheets:', list(all_sheets.keys()))

Saved: ORDERS.csv
Saved: ORDER_DETAILS.csv
Saved: CUSTOMERS.csv
Saved: PRODUCTS.csv
Saved: CATEGORIES.csv
Saved: STORES.csv
Saved: CHANNELS.csv
Saved: KPI.csv
Sheets: ['ORDERS', 'ORDER_DETAILS', 'CUSTOMERS', 'PRODUCTS', 'CATEGORIES', 'STORES', 'CHANNELS', 'KPI']


### **Step 3: Metrics & Export**

#### **1. Total Revenue**

In [28]:
o_d = pd.read_csv(created_files[1]) # Order_Details.csv
o_d.info()

<class 'pandas.DataFrame'>
RangeIndex: 85 entries, 0 to 84
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   order_detail_id   85 non-null     str  
 1   order_id          85 non-null     str  
 2   product_id        85 non-null     str  
 3   quantity          85 non-null     int64
 4   unit_price        85 non-null     int64
 5   discount_percent  85 non-null     int64
dtypes: int64(3), str(3)
memory usage: 4.1 KB


In [ ]:
# create column revenue by category
o_d['Revenue'] = o_d['quantity'] * o_d['unit_price'] * (1 - o_d['discount_percent']/100)
o_d.head()

,order_detail_id,order_id,product_id,quantity,unit_price,discount_percent,Revenue
0,OD001,ORD001,PRD001,2,299000,0,598000.0
1,OD002,ORD001,PRD005,1,199000,0,199000.0
2,OD003,ORD002,PRD002,1,599000,10,539100.0
3,OD004,ORD002,PRD014,2,299000,10,538200.0
4,OD005,ORD003,PRD003,3,899000,5,2562150.0


In [27]:
# total revenue
total_revenue = o_d['Revenue'].sum()
print(f'Total Revenue: ${total_revenue:,.2f}')

Total Revenue: $66,272,500.00


In [ ]:
# xuất ra file csv 
o_d.to_csv('data/csv/Order_Details.csv', index=False)

# **END**